In [ ]:
from eventflow.data import load_train, make_train_examples
from eventflow.validation import add_folds, split
from eventflow.config import EVENT_CLASSES, TUNING_FOLD
from eventflow.features import make_features, CATEGORICAL

train = load_train()
examples = add_folds(make_train_examples(train))
X = make_features(train)
print("feature table:", X.shape)        # expect (1203963, 79)
X.iloc[95:100, :]

feature table: (1203963, 79)


,ev_lag0,sev_lag0,ev_lag1,sev_lag1,ev_lag2,sev_lag2,run_length,cnt5_COMPONENT_REPLACED,cnt5_CONTROL_ALERT,cnt5_COOLING_ACTIVATED,...,share_LOAD_INCREASE,share_MAINTENANCE_CHECK,share_PRESSURE_WARNING,share_RECOVERY,share_ROUTINE_CHECK,share_SENSOR_RESET,share_TEMP_WARNING,share_VIBRATION_WARNING,mean_severity_so_far,events_so_far
95,7,0.0,3,2.0,2,0.0,1,0,0,1,...,0.041667,0.145833,0.031250,0.208333,0.020833,0.010417,0.125000,0.197917,1.072917,96
96,2,0.0,7,0.0,3,2.0,1,0,0,2,...,0.041237,0.144330,0.030928,0.206186,0.020619,0.010309,0.123711,0.195876,1.061856,97
97,2,0.0,2,0.0,7,0.0,2,0,0,3,...,0.040816,0.142857,0.030612,0.204082,0.020408,0.010204,0.122449,0.193878,1.051020,98
98,9,1.0,2,0.0,2,0.0,1,0,0,2,...,0.040404,0.141414,0.030303,0.212121,0.020202,0.010101,0.121212,0.191919,1.050505,99
99,12,1.0,9,1.0,2,0.0,1,0,0,2,...,0.040000,0.140000,0.030000,0.210000,0.020000,0.010000,0.130000,0.190000,1.050000,100


In [2]:
row = int(examples["last_row"].iloc[1234])
seq_id = train.loc[row, "sequence_id"]
seq_start = train.index[train["sequence_id"] == seq_id][0]
history = train.loc[seq_start:row, "event_type"].tolist()

by_hand = {
    "ev_lag0": EVENT_CLASSES.index(history[-1]),
    "ev_lag2": EVENT_CLASSES.index(history[-3]),
    "cnt30_TEMP_WARNING": history[-30:].count("TEMP_WARNING"),
    "cnt5_RECOVERY": history[-5:].count("RECOVERY"),
}
run = 1
while run < len(history) and history[-1 - run] == history[-1]:
    run += 1
by_hand["run_length"] = run

for name, value in by_hand.items():
    assert X.loc[row, name] == value, (name, X.loc[row, name], value)
print("Hand check passed:", by_hand)

Hand check passed: {'ev_lag0': 2, 'ev_lag2': 12, 'cnt30_TEMP_WARNING': 1, 'cnt5_RECOVERY': 2, 'run_length': 2}


In [3]:
# NO-FUTURE TEST (leakage check)
# e.g. a diary of 250 events, cut at row 125:
#   full = make_features(rows 0..249) → features at row 125
#   cut  = make_features(rows 0..125) → features at row 125
#   same      → no feature used rows 126..249 → no leakage 
#   different → some feature peeked at the future → leakage 
lengths = train.groupby("sequence_id").size()
seq_id = lengths.idxmax()                                  # the longest machine
seq = train[train["sequence_id"] == seq_id].reset_index(drop=True)
full = make_features(seq)                                  # the whole diary

for c in [0, 10, 89, len(seq) // 2, len(seq) - 2]:         # several cut points
    cut = make_features(seq.iloc[:c + 1])                  # the diary cut after row c
    assert full.iloc[c].equals(cut.iloc[c]), c
print(f"No-future test passed on {seq_id} ({len(seq)} events): features at a cut do not change when later events are removed")

No-future test passed on TR00160 (250 events): features at a cut do not change when later events are removed


In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from eventflow.config import N_CLASSES, RANDOM_STATE
from eventflow.features import FEATURE_GROUPS
from eventflow.validation import evaluate

# Fold 0 is used ONLY for the score. Early stopping needs data of its own, otherwise it picks the
# best round on fold 0 and the fold-0 score is a bit too good. So hold back 10% of the
# training-fold machines (whole sequences, never split) as an early-stopping set.
tr, va = split(examples, TUNING_FOLD)
rng = np.random.default_rng(RANDOM_STATE)
tr_ids = tr["sequence_id"].unique()
stop_ids = rng.choice(tr_ids, size=len(tr_ids) // 10, replace=False)
is_stop = tr["sequence_id"].isin(stop_ids).to_numpy()
fit_part = tr[~is_stop].sample(150_000, random_state=RANDOM_STATE)   # smaller, so it runs in a few minutes
stop_part = tr[is_stop]
print(f"fit: {len(fit_part):,} examples | early stopping: {len(stop_part):,} examples "
      f"({len(stop_ids)} machines) | score (fold {TUNING_FOLD}): {len(va):,} examples")

# The columns each feature group makes (computed on a few rows just to read the names)
group_cols = {g: list(f(train.head(300)).columns) for g, f in FEATURE_GROUPS.items()}
cat_type = pd.CategoricalDtype(categories=range(-1, N_CLASSES))   # -1 = before the start of the sequence

def rows_at_cut(part, cols):
    """The feature row at each example's cut, with the event-lag columns as categories."""
    return X.loc[part["last_row"], cols].astype({c: cat_type for c in CATEGORICAL if c in cols})

results = {}
for groups in [["recent", "counts"], ["recent", "counts", "since"], ["recent", "counts", "since", "habits"]]:
    cols = [c for g in groups for c in group_cols[g]]
    model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63,
                               random_state=RANDOM_STATE, verbose=-1)
    model.fit(rows_at_cut(fit_part, cols), fit_part["target"],
              eval_X=rows_at_cut(stop_part, cols), eval_y=stop_part["target"],
              callbacks=[lgb.early_stopping(30, verbose=False)])
    assert len(model.classes_) == N_CLASSES
    scores = evaluate(va, model.predict_proba(rows_at_cut(va, cols)))
    results[" + ".join(groups)] = {"n_features": len(cols), "best_round": model.best_iteration_,
                                   "logloss": scores["logloss"], "logloss_mimic": scores["logloss_mimic"]}

# logloss       = score on ALL validation cuts (prefix lengths 90-220): the main score
# logloss_mimic = score on only the one cut per machine made like test.csv (length - 30)
pd.DataFrame(results).T.round(4)

fit: 150,000 examples | early stopping: 47,974 examples (520 machines) | score (fold 0): 117,311 examples


,n_features,best_round,logloss,logloss_mimic
recent + counts,49.0,106.0,1.5439,1.5410
recent + counts + since,63.0,111.0,1.5185,1.5061


Quick check (fold 0, same 150k training sample, early stopping on 520 held-back training machines):
groups 1–2 → 1.544 (mimic 1.541); groups 1–3 → 1.519 (mimic 1.506), so group 3 (events since) helps (−0.025).

Early stopping now uses its own data instead of fold 0. The scores barely moved (the earlier run gave 1.544 and 1.518), so the old numbers were only very slightly optimistic.

In [5]:
import numpy as np

row = int(examples["last_row"].iloc[1234])
seq_id = train.loc[row, "sequence_id"]
seq_start = train.index[train["sequence_id"] == seq_id][0]
history = train.loc[seq_start:row, "event_type"].tolist()

def events_ago(history, event):
    """Slow, obvious version: walk backwards until we find the event."""
    for steps, e in enumerate(reversed(history)):
        if e == event:
            return steps
    return np.nan                                   # never happened in this sequence

for event in EVENT_CLASSES:
    fast, slow = X.loc[row, f"since_{event}"], events_ago(history, event)
    assert (np.isnan(fast) and np.isnan(slow)) or fast == slow, (event, fast, slow)
print("Group 3 hand check passed:", {e: events_ago(history, e) for e in EVENT_CLASSES[:4]})

Group 3 hand check passed: {'COMPONENT_REPLACED': 10, 'CONTROL_ALERT': nan, 'COOLING_ACTIVATED': 0, 'COOLING_FAULT': 82}


In [6]:
row = int(examples["last_row"].iloc[1234])
seq_id = train.loc[row, "sequence_id"]
seq_start = train.index[train["sequence_id"] == seq_id][0]
history = train.loc[seq_start:row, "event_type"].tolist()
sev_history = train.loc[seq_start:row, "severity"].tolist()

by_hand = {
    "share_RECOVERY": history.count("RECOVERY") / len(history),
    "share_TEMP_WARNING": history.count("TEMP_WARNING") / len(history),
    "mean_severity_so_far": sum(sev_history) / len(sev_history),
    "events_so_far": len(history),
}
for name, value in by_hand.items():
    assert np.isclose(X.loc[row, name], value), (name, X.loc[row, name], value)
print("Group 4 hand check passed:", {k: round(v, 3) for k, v in by_hand.items()})

Group 4 hand check passed: {'share_RECOVERY': 0.371, 'share_TEMP_WARNING': 0.124, 'mean_severity_so_far': 0.619, 'events_so_far': 97}
